In [2]:
#!pip uninstall keras -y
#!pip install tf-keras

Found existing installation: keras 3.10.0
Uninstalling keras-3.10.0:
  Successfully uninstalled keras-3.10.0
Defaulting to user installation because normal site-packages is not writeable
  Using cached keras-3.10.0-py3-none-any.whl.metadata (6.0 kB)
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   --------------------- ------------------ 0.9/1.7 MB 29.7 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 27.7 MB/s eta 0:00:00
Using cached keras-3.10.0-py3-none-any.whl (1.4 MB)



[notice] A new release of pip is available: 24.1.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import gradio as gr
import pandas as pd
import numpy as np
import faiss
import json
import os
import csv
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")


In [8]:
# === Load FAISS index and chunked data ===
faiss_path = r"C:\Users\manjo\Downloads\Project\faiss_index_PaddleOCR.index"
csv_path = r"C:\Users\manjo\Downloads\Project\extracted_text_PaddleOCR2\cleaned_text_PaddleOCR2_hybrid\chunked_text_PaddleOCR_hybrid.csv"
index = faiss.read_index(faiss_path)
df_chunks = pd.read_csv(csv_path)


In [9]:
# === Load embedding model ===
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


In [13]:

# === Cache and Feedback files ===
CACHE_FILE = "cache.json"
FEEDBACK_FILE = "feedback_log.csv"

In [15]:
# === Clear old cache on run ===
with open(CACHE_FILE, "w", encoding="utf-8") as f:
    json.dump({}, f)

In [16]:
## === Cache Utils ===
def load_cache():
    return json.load(open(CACHE_FILE, "r", encoding="utf-8")) if os.path.exists(CACHE_FILE) else {}

def save_cache(cache):
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=4)

def save_feedback(query, answer, source, feedback):
    data = {"query": query, "answer": answer, "source": source, "feedback": feedback}
    file_exists = os.path.isfile(FEEDBACK_FILE)
    with open(FEEDBACK_FILE, "a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=data.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(data)

In [17]:
# === Top-k chunk search ===
def search_top_k(query, k=2):
    query_embedding = embedding_model.encode([query])
    distances, indices = index.search(np.array(query_embedding).astype("float32"), k)
    return df_chunks.iloc[indices[0]][["chunk_id", "filename", "chunk_text"]]

In [18]:
# === Build prompt for LLM ===
def build_prompt(chunks, question):
    context = "\n".join(chunks["chunk_text"].tolist())
    prompt = f"""You are a helpful assistant. Use the context below to answer the user's question.\n\n### Context\n{context}\n\n### Question\n{question}\n\n### Answer\n"""
    return prompt

In [19]:
# === Generate LLM Answer ===
def generate_answer(chunks, question):
    prompt = build_prompt(chunks, question)
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids
    output = model.generate(input_ids, max_new_tokens=200, do_sample=True)
    return tokenizer.decode(output[0], skip_special_tokens=True).split("### Answer")[-1].strip()

In [20]:
# === Chatbot UI logic ===
cache = load_cache()

def chatbot_ui(query, history):
    if query in cache:
        answer = cache[query]["answer"]
        source = cache[query]["source"]
    else:
        chunks = search_top_k(query)
        answer = generate_answer(chunks, query)
        source = chunks.iloc[0]["filename"]
        cache[query] = {"answer": answer, "source": source}
        save_cache(cache)

    display_answer = f"*Answer:* {answer}\n*Source:* {source}"
    history.append((query, display_answer))
    return history, query, answer, source

def feedback_fn(query, answer, source, feedback, history):
    save_feedback(query, answer, source, feedback)
    return history + [[f"✅ Feedback received: {feedback.upper()}", ""]]

In [24]:
import gradio as gr

# === Load cache ===
cache = load_cache()

# === Define chatbot logic ===
def chatbot_ui(query, history):
    if history is None:
        history = []
    if query in cache:
        answer = cache[query]["answer"]
        source = cache[query]["source"]
    else:
        chunks = search_top_k(query)
        source = chunks.iloc[0]["filename"]
        answer = generate_answer(chunks, query)
        save_cache({query: {"answer": answer, "source": source}})
    history.append((query, f"*Answer:* {answer}\n\n*Source:* {source}"))
    return history, query, answer, source

# === Feedback function ===
def feedback_fn(query, answer, source, feedback, history):
    if history is None:
        history = []
    save_feedback(query, answer, source, feedback)
    history.append((f"✅ Feedback received: {feedback.upper()}", ""))
    return history

# === UI Layout ===
with gr.Blocks(title="🤖 Chatter - RAG Chatbot") as demo:
    gr.Markdown("## 🤖 *Chatter:* RAG-based Chatbot with RLHF")

    chatbot = gr.Chatbot()
    query = gr.Textbox(label="Ask something...")
    answer_box = gr.Textbox(visible=False)
    source_box = gr.Textbox(visible=False)

    with gr.Row():
        btn_submit = gr.Button("Submit")
        btn_good = gr.Button("👍")
        btn_bad = gr.Button("👎")

    state_query = gr.State()
    state_answer = gr.State()
    state_source = gr.State()
    state_history = gr.State([])

    # Submit button logic
    btn_submit.click(fn=chatbot_ui,
                     inputs=[query, state_history],
                     outputs=[chatbot, state_query, state_answer, state_source])

    # Feedback buttons
    btn_good.click(fn=feedback_fn,
                   inputs=[state_query, state_answer, state_source, gr.Textbox(value="good"), chatbot],
                   outputs=[chatbot])

    btn_bad.click(fn=feedback_fn,
                  inputs=[state_query, state_answer, state_source, gr.Textbox(value="bad"), chatbot],
                  outputs=[chatbot])

    demo.launch()

C:\Users\manjo\AppData\Local\Temp\ipykernel_12112\131857000.py:33: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
